# 07. Entrenamiento de Modelos Híbridos (RandomForest / XGBoost)

**Objetivo:** entrenar y comparar clasificadores con las features híbridas exportadas por 06.
**Entradas:** `features_core.parquet`, `features_py.parquet`, `data/splits/*_indices.csv`.
**Salidas:** `data/outputs/train_<run_id>/comparacion_modelos_<split>.csv`, `predicciones_*_<split>.csv`, `figures/*.png`, `modelo_*.joblib`, `resumen_entrenamiento.json`, `resumen_ablacion.json`.
**Notebook anterior:** `notebooks/pipeline/06_ingenieria_features_hibridas.ipynb`.
**Notebook siguiente:** `notebooks/pipeline/08_resultados_hibrido_vs_lineas_base.ipynb` y `notebooks/pipeline/09b_cierre_modelos_dev.ipynb`.


## Técnicas, herramientas y librerías de esta etapa

- **Técnica principal:** entrenamiento tabular con ablaciones controladas sobre familias de features.
- **Herramientas/librerías:** `scikit-learn` (`RandomForestClassifier`, `RandomizedSearchCV`, `GroupKFold`, métricas), `xgboost` (`XGBClassifier`), `joblib`, `matplotlib`, `pandas`, `numpy`.
- **Por qué es adecuada aquí:** el espacio de entrada ya no es texto crudo sino una tabla heterogénea de features simbólicas y contextuales. Para ese tipo de entrada, `RandomForest` y `XGBoost` son opciones razonables, permiten ablaciones claras y exportan artefactos interpretables.
- **Limitación:** no modelan la secuencia textual original; dependen por completo de la calidad de las features construidas en 06.
- **Alternativa si se quisiera priorizar otra cosa:** modelos lineales o regularizados serían más simples, pero suelen capturar peor interacciones no lineales entre bloques. Modelos secuenciales end-to-end no serían comparables con el híbrido actual.


## Criterio metodológico
- Comparación `core` vs `py` como ablación controlada.
- Comparación de clasificadores: `RandomForest` vs `XGBoost`.
- Se permite `RandomizedSearchCV` opcional para tuning.
- Split de evaluación configurable (`dev` para ajuste, `test` para reporte final).


## Banderas de ablación y contrato de comparación (07)

La comparación entre variantes se hace sobre el mismo universo de evaluación (`TRAIN_EVAL_ON`) y el mismo conjunto de features de entrada (`TRAIN_FEATURE_RUN_ID_CORE`, `TRAIN_FEATURE_RUN_ID_PY`).

**Controles de ejecución:**
- `TRAIN_MODELS`: `xgb`, `rf`.
- `TRAIN_PROFILES`: `core`, `py`.
- `TRAIN_SEED`: semilla trazable por corrida.

**Controles de familias de features:**
- `TRAIN_USE_LLM = 1 | 0`: en `0` reemplaza `feat_*` por su contraparte `rule_*` cuando existe.
- `TRAIN_USE_BETO = 1 | 0`: alias legacy para activar/desactivar embeddings contextuales.
- `TRAIN_USE_CONTEXT = 1 | 0`: controla embeddings contextuales (`ctx_*` y legacy `beto_*`).
- `TRAIN_USE_TEMPLATE = 1 | 0`: controla `feat_had_template_block`.
- `TRAIN_USE_FEAT = 1 | 0`: conserva o elimina `feat_*`.
- `TRAIN_USE_RULES = 1 | 0`: controla `rule_*` (excepto medicación).
- `TRAIN_USE_MEDICATION = 1 | 0`: controla `rule_medication_*`.
- `TRAIN_USE_SENTIMENT = 1 | 0`: conserva o elimina `sent_*`.

**Ablación fina adicional:**
- `TRAIN_DROP_COLUMNS`, `TRAIN_DROP_PREFIXES`, `TRAIN_KEEP_PREFIXES` para variantes puntuales.

Cada corrida exporta `resumen_ablacion.json` y `detalle_columnas_ablacion.json` para trazabilidad exacta de columnas incluidas/excluidas.


In [1]:
import json
import os
import re
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.preprocessing import LabelEncoder

from utils_shared import setup_paths, load_splits, ensure_dir, guess_label_col, guess_patient_id_col, guess_text_col

warnings.filterwarnings('ignore')

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception as e:
    HAS_XGB = False
    print('Aviso: xgboost no disponible:', e)

paths = setup_paths()
BASE_PATH = paths['BASE_PATH']
DATA_PATH = paths['DATA_PATH']
SPLITS_PATH = paths['SPLITS_PATH']
PROCESSED_PATH = paths['PROCESSED_PATH']
OUTPUTS_PATH = paths['OUTPUTS_PATH']

print('HAS_XGB:', HAS_XGB)


HAS_XGB: True


In [2]:
# Configuración de entrenamiento y evaluación
# Búsqueda de hiperparámetros opcional con `RandomizedSearchCV` para modelos tabulares.

def parse_list_env(var_name: str, default: str = "") -> list[str]:
    raw = os.getenv(var_name, default)
    if raw is None:
        return []
    vals = [x.strip() for x in str(raw).split(',') if x.strip()]
    return vals


def parse_flag(var_name: str, default: str = '1') -> bool:
    return os.getenv(var_name, default).strip() == '1'

EVAL_ON = os.getenv('TRAIN_EVAL_ON', 'dev').strip().lower()  # 'dev' o 'test'
if EVAL_ON not in {'dev', 'test'}:
    raise ValueError(f"TRAIN_EVAL_ON inválido: {EVAL_ON}. Use 'dev' o 'test'.")

FEATURE_RUN_ID_CORE = os.getenv('TRAIN_FEATURE_RUN_ID_CORE') or None
FEATURE_RUN_ID_PY = os.getenv('TRAIN_FEATURE_RUN_ID_PY') or None

USE_RANDOMIZED_SEARCH = parse_flag('TRAIN_USE_RANDOM_SEARCH', '1')
USE_XGB = parse_flag('TRAIN_USE_XGB', '1')
REQUIRE_XGB = parse_flag('TRAIN_REQUIRE_XGB', '0')

N_ITER_SEARCH = int(os.getenv('TRAIN_N_ITER_SEARCH', '25'))
CV_FOLDS = int(os.getenv('TRAIN_CV_FOLDS', '3'))
TRAIN_SEED = int(os.getenv('TRAIN_SEED', os.getenv('TRAIN_RANDOM_SEED', '42')))
RANDOM_SEED = TRAIN_SEED
N_JOBS = int(os.getenv('TRAIN_N_JOBS', '-1'))

MODELS_ENV = [m.lower() for m in parse_list_env('TRAIN_MODELS', 'xgb,rf')]
PROFILES_ENV = [p.lower() for p in parse_list_env('TRAIN_PROFILES', 'core,py')]
VALID_MODELS = {'xgb', 'rf'}
VALID_PROFILES = {'core', 'py'}
MODELS_SELECTED = [m for m in ['rf', 'xgb'] if m in set(MODELS_ENV) & VALID_MODELS]
PROFILES_SELECTED = [p for p in ['core', 'py'] if p in set(PROFILES_ENV) & VALID_PROFILES]
if not PROFILES_SELECTED:
    raise ValueError('TRAIN_PROFILES no contiene valores válidos (core,py).')

# Compatibilidad con TRAIN_USE_XGB
if not USE_XGB and 'xgb' in MODELS_SELECTED:
    MODELS_SELECTED = [m for m in MODELS_SELECTED if m != 'xgb']

if not MODELS_SELECTED:
    raise ValueError('No quedaron modelos para entrenar. Revisa TRAIN_MODELS y TRAIN_USE_XGB.')

TRAIN_DROP_COLUMNS = parse_list_env('TRAIN_DROP_COLUMNS', '')
TRAIN_DROP_PREFIXES = parse_list_env('TRAIN_DROP_PREFIXES', '')
TRAIN_KEEP_PREFIXES = parse_list_env('TRAIN_KEEP_PREFIXES', '')

TRAIN_USE_BETO = parse_flag('TRAIN_USE_BETO', '1')
TRAIN_USE_CONTEXT = parse_flag('TRAIN_USE_CONTEXT', '1' if TRAIN_USE_BETO else '0')
TRAIN_USE_TEMPLATE = parse_flag('TRAIN_USE_TEMPLATE', '1')
TRAIN_USE_FEAT = parse_flag('TRAIN_USE_FEAT', '1')
TRAIN_USE_RULES = parse_flag('TRAIN_USE_RULES', '1')
TRAIN_USE_MEDICATION = parse_flag('TRAIN_USE_MEDICATION', '1')
TRAIN_USE_SENTIMENT = parse_flag('TRAIN_USE_SENTIMENT', '1')
TRAIN_USE_LLM = parse_flag('TRAIN_USE_LLM', '1')

VARIANT_NAME = os.getenv('TRAIN_VARIANT_NAME', '').strip() or 'variante_base'

RUN_ID = os.getenv('TRAIN_RUN_ID') or pd.Timestamp.now().strftime('train_%Y%m%d_%H%M%S')
OUT_DIR = ensure_dir(OUTPUTS_PATH / RUN_ID)
FIG_DIR = ensure_dir(OUT_DIR / 'figures')

print('OUT_DIR:', OUT_DIR)
print('EVAL_ON:', EVAL_ON)
print('VARIANT_NAME:', VARIANT_NAME)
print('USE_RANDOMIZED_SEARCH:', USE_RANDOMIZED_SEARCH)
print('MODELS_SELECTED:', MODELS_SELECTED)
print('PROFILES_SELECTED:', PROFILES_SELECTED)
print('TRAIN_SEED:', TRAIN_SEED)
print('FLAGS -> llm:', TRAIN_USE_LLM, '| context:', TRAIN_USE_CONTEXT, '| beto_legacy:', TRAIN_USE_BETO, '| template:', TRAIN_USE_TEMPLATE, '| feat:', TRAIN_USE_FEAT,
      '| rules:', TRAIN_USE_RULES, '| medication:', TRAIN_USE_MEDICATION, '| sentiment:', TRAIN_USE_SENTIMENT)
print('DROP_COLUMNS:', TRAIN_DROP_COLUMNS)
print('DROP_PREFIXES:', TRAIN_DROP_PREFIXES)
print('KEEP_PREFIXES:', TRAIN_KEEP_PREFIXES)

if 'xgb' in MODELS_SELECTED and REQUIRE_XGB and not HAS_XGB:
    raise ImportError('Se solicitó XGBoost (TRAIN_REQUIRE_XGB=1), pero no está disponible en el entorno.')


OUT_DIR: /Users/manuelnunez/Projects/psych-phenotyping-paraguay/data/outputs/train_20260310_093418
EVAL_ON: dev
USE_RANDOMIZED_SEARCH: True
USE_XGB: True
REQUIRE_XGB: False


In [3]:
def _latest_feature_run(suffix: str):
    cand = sorted(PROCESSED_PATH.glob(f'fe_*_{suffix}'))
    return cand[-1].name if cand else None

FEATURE_RUN_ID_CORE = FEATURE_RUN_ID_CORE or _latest_feature_run('core')
FEATURE_RUN_ID_PY = FEATURE_RUN_ID_PY or _latest_feature_run('py')

if FEATURE_RUN_ID_CORE is None or FEATURE_RUN_ID_PY is None:
    raise FileNotFoundError('No se detectaron corridas de features en data/processed/fe_*_core|py')

print('FEATURE_RUN_ID_CORE:', FEATURE_RUN_ID_CORE)
print('FEATURE_RUN_ID_PY  :', FEATURE_RUN_ID_PY)


def load_features(run_id: str, suffix: str) -> pd.DataFrame:
    fpath = PROCESSED_PATH / run_id / f'features_{suffix}.parquet'
    if not fpath.exists():
        raise FileNotFoundError(f'No existe: {fpath}')
    return pd.read_parquet(fpath)


def _base_feature_run_id(run_id: str) -> str:
    if run_id.endswith('_core'):
        return run_id[:-5]
    if run_id.endswith('_py'):
        return run_id[:-3]
    return run_id


def load_feature_config(run_id: str) -> dict:
    cfg_path = PROCESSED_PATH / f"{_base_feature_run_id(run_id)}_config.json"
    if not cfg_path.exists():
        return {}
    try:
        with open(cfg_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        print('Aviso: no se pudo leer config de features:', cfg_path, '|', e)
        return {}


FEATURE_CONFIG = load_feature_config(FEATURE_RUN_ID_CORE) or load_feature_config(FEATURE_RUN_ID_PY)
df_core = load_features(FEATURE_RUN_ID_CORE, 'core')
df_py = load_features(FEATURE_RUN_ID_PY, 'py')

# Compatibilidad con corridas legacy sin config.json:
# - LLM: por defecto se asume activo en el híbrido histórico (late fusion).
# - Sentimiento: se infiere por presencia de columnas sent_*.
FEATURE_LLM_ACTIVO = bool(FEATURE_CONFIG.get('llm_activo', True))
FEATURE_SENT_ACTIVO = bool(FEATURE_CONFIG.get('sentimiento_activo', any(c.startswith('sent_') for c in df_py.columns)))

print('core shape:', df_core.shape)
print('py   shape:', df_py.shape)

label_col = guess_label_col(df_py)
pid_col = guess_patient_id_col(df_py)
text_col = guess_text_col(df_py)
if label_col is None:
    raise ValueError('No se detectó columna de etiqueta en features.')

print('label_col:', label_col)
print('patient_id_col:', pid_col)
print('text_col:', text_col)


def _infer_context_prefixes(df: pd.DataFrame) -> list[str]:
    pref = set()
    for c in df.columns:
        m = re.match(r'^(ctx_[a-z0-9_]+_)\d+$', str(c))
        if m:
            pref.add(m.group(1))
        elif str(c).startswith('beto_'):
            pref.add('beto_')
    return sorted(pref)


def _infer_text_backbone(cfg: dict, df: pd.DataFrame) -> str:
    if cfg.get('text_backbone'):
        return str(cfg.get('text_backbone'))
    prefs = _infer_context_prefixes(df)
    if not prefs:
        return 'none'
    if prefs[0] == 'beto_':
        return 'beto'
    return prefs[0].replace('ctx_', '').rstrip('_')

FEATURE_CONTEXT_PREFIXES = _infer_context_prefixes(df_py)
FEATURE_TEXT_BACKBONE = _infer_text_backbone(FEATURE_CONFIG, df_py)
FEATURE_CONTEXT_ACTIVO = bool(FEATURE_CONFIG.get('contexto_activo', len(FEATURE_CONTEXT_PREFIXES) > 0))

print('feature_text_backbone:', FEATURE_TEXT_BACKBONE)
print('feature_context_prefixes:', FEATURE_CONTEXT_PREFIXES)


FEATURE_RUN_ID_CORE: fe_20260310_082139_core
FEATURE_RUN_ID_PY  : fe_20260310_082139_py
core shape: (1835, 958)
py   shape: (1835, 962)
label_col: etiqueta
patient_id_col: patient_id
text_col: texto


In [4]:
# Particiones patient-level
_, train_ids, dev_ids, test_ids = load_splits(SPLITS_PATH)
train_ids = set(train_ids)
dev_ids = set(dev_ids)
test_ids = set(test_ids)


def subset_by_split(df: pd.DataFrame, split: str) -> pd.DataFrame:
    idx = {'train': train_ids, 'dev': dev_ids, 'test': test_ids}[split]
    return df[df['row_id'].isin(idx)].copy()


def build_xy(df: pd.DataFrame):
    y = df[label_col].astype(str)
    meta = df[['row_id', label_col]].copy().rename(columns={label_col: 'y_true'})

    if pid_col is not None and pid_col in df.columns:
        grupos = df[pid_col].astype(str)
    else:
        grupos = df['row_id'].astype(str)

    drop_cols = {'row_id', label_col}
    if pid_col is not None and pid_col in df.columns:
        drop_cols.add(pid_col)
    if text_col in df.columns:
        drop_cols.add(text_col)

    X = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

    # Evita fallas por columnas no numéricas (ej. sent_label)
    non_num = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    if non_num:
        X = X.drop(columns=non_num)

    return X, y, meta, grupos


def _drop_by_prefix(cols: list[str], prefixes: list[str]) -> set[str]:
    out = set()
    for c in cols:
        for p in prefixes:
            if c.startswith(p):
                out.add(c)
                break
    return out


def aplicar_ablacion_columnas(X_in: pd.DataFrame, profile: str):
    cols_all = list(X_in.columns)
    selected = set(cols_all)
    motivo_drop: dict[str, str] = {}

    def drop_cols(cols, reason):
        for c in cols:
            if c in selected:
                selected.remove(c)
                motivo_drop[c] = reason

    # 1) Keep prefixes (si se define, restringe el universo inicial)
    if TRAIN_KEEP_PREFIXES:
        keep = {c for c in cols_all if any(c.startswith(p) for p in TRAIN_KEEP_PREFIXES)}
        drop_cols([c for c in cols_all if c not in keep], 'keep_prefixes')

    # 2) Toggles principales
    if not TRAIN_USE_CONTEXT:
        drop_cols(_drop_by_prefix(list(selected), ['beto_', 'ctx_']), 'toggle_context_off')

    if not TRAIN_USE_TEMPLATE:
        drop_cols(['feat_had_template_block'], 'toggle_template_off')

    if not TRAIN_USE_FEAT:
        drop_cols(_drop_by_prefix(list(selected), ['feat_']), 'toggle_feat_off')

    if not TRAIN_USE_RULES:
        drop_cols(
            [c for c in list(selected) if c.startswith('rule_') and not c.startswith('rule_medication_')],
            'toggle_rules_off'
        )

    if not TRAIN_USE_MEDICATION:
        drop_cols(_drop_by_prefix(list(selected), ['rule_medication_']), 'toggle_medication_off')

    if not TRAIN_USE_SENTIMENT:
        drop_cols(_drop_by_prefix(list(selected), ['sent_']), 'toggle_sentiment_off')

    # 3) Drops explícitos
    if TRAIN_DROP_PREFIXES:
        drop_cols(_drop_by_prefix(list(selected), TRAIN_DROP_PREFIXES), 'drop_prefixes')

    if TRAIN_DROP_COLUMNS:
        drop_cols([c for c in TRAIN_DROP_COLUMNS if c in selected], 'drop_columns')

    selected_cols = sorted(selected)
    if not selected_cols:
        raise ValueError(f'[{profile}] No quedaron columnas tras aplicar la ablación.')

    X_out = X_in.reindex(columns=selected_cols, fill_value=0).copy()

    resumen = {
        'profile': profile,
        'n_cols_antes': len(cols_all),
        'n_cols_despues': len(selected_cols),
        'n_cols_drop': len(cols_all) - len(selected_cols),
        'columns_selected': selected_cols,
        'columns_dropped': motivo_drop,
    }
    return X_out, resumen




def aplicar_modo_llm(X_in: pd.DataFrame) -> pd.DataFrame:
    """Controla contribución LLM en feat_* sin cambiar el universo de filas.

    - TRAIN_USE_LLM=1: mantiene feat_* tal como vienen del feature run.
    - TRAIN_USE_LLM=0: reemplaza feat_* por su contraparte rule_* cuando existe,
      y feat_niega_* por niega_* cuando existe.
    """
    if TRAIN_USE_LLM:
        return X_in

    X = X_in.copy()

    # Síntomas positivos
    feat_cols = [c for c in X.columns if c.startswith('feat_') and not c.startswith('feat_niega_') and c != 'feat_had_template_block']
    for fc in feat_cols:
        ph = fc.replace('feat_', '', 1)
        rc = f'rule_{ph}'
        if rc in X.columns:
            X[fc] = X[rc]

    # Negaciones por síntoma
    feat_neg_cols = [c for c in X.columns if c.startswith('feat_niega_')]
    for fnc in feat_neg_cols:
        ph = fnc.replace('feat_niega_', '', 1)
        nc = f'niega_{ph}'
        if nc in X.columns:
            X[fnc] = X[nc]

    return X

profile_data = {}
feature_logs = {}

df_map = {'core': df_core, 'py': df_py}
for profile in PROFILES_SELECTED:
    dfi = df_map[profile]
    d_tr = subset_by_split(dfi, 'train')
    d_dev = subset_by_split(dfi, 'dev')
    d_te = subset_by_split(dfi, 'test')

    X_tr_raw, y_tr, M_tr, G_tr = build_xy(d_tr)
    X_dev_raw, y_dev, M_dev, _ = build_xy(d_dev)
    X_te_raw, y_te, M_te, _ = build_xy(d_te)

    X_tr_raw = aplicar_modo_llm(X_tr_raw)
    X_dev_raw = aplicar_modo_llm(X_dev_raw)
    X_te_raw = aplicar_modo_llm(X_te_raw)

    X_tr, flog = aplicar_ablacion_columnas(X_tr_raw, profile)
    X_dev = X_dev_raw.reindex(columns=X_tr.columns, fill_value=0)
    X_te = X_te_raw.reindex(columns=X_tr.columns, fill_value=0)

    profile_data[profile] = {
        'X_tr': X_tr,
        'y_tr': y_tr,
        'M_tr': M_tr,
        'G_tr': G_tr,
        'X_dev': X_dev,
        'y_dev': y_dev,
        'M_dev': M_dev,
        'X_te': X_te,
        'y_te': y_te,
        'M_te': M_te,
    }
    feature_logs[profile] = flog

    print(f"{profile.upper()} X train: {X_tr.shape} | y: {y_tr.value_counts().to_dict()}")

# Guardar log detallado de columnas incluidas/excluidas
with open(OUT_DIR / 'detalle_columnas_ablacion.json', 'w', encoding='utf-8') as f:
    json.dump(feature_logs, f, ensure_ascii=False, indent=2)
print('Detalle de columnas de ablación:', OUT_DIR / 'detalle_columnas_ablacion.json')


CORE X train: (1107, 954) | y: {'depresion': 749, 'ansiedad': 358}
PY   X train: (1107, 958) | y: {'depresion': 749, 'ansiedad': 358}


In [5]:
def _resolve_cv(groups):
    """Configura CV por grupos cuando hay suficiente diversidad de pacientes."""
    if groups is None:
        return CV_FOLDS, None

    n_groups = pd.Series(groups).nunique()
    if n_groups < 2:
        return CV_FOLDS, None

    n_splits = min(CV_FOLDS, n_groups)
    if n_splits < 2:
        return CV_FOLDS, None

    return GroupKFold(n_splits=n_splits), groups


def entrenar_rf(X_train, y_train, groups=None):
    base = RandomForestClassifier(
        n_estimators=500,
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS,
        class_weight='balanced',
    )

    if not USE_RANDOMIZED_SEARCH:
        base.fit(X_train, y_train)
        return base, {'modo': 'sin_tuning'}, np.nan

    cv, fit_groups = _resolve_cv(groups)

    param_dist = {
        'n_estimators': [300, 500, 800],
        'max_depth': [None, 8, 12, 16],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2', None],
    }
    search = RandomizedSearchCV(
        base,
        param_distributions=param_dist,
        n_iter=N_ITER_SEARCH,
        scoring='f1_macro',
        cv=cv,
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS,
        verbose=1,
    )
    if fit_groups is None:
        search.fit(X_train, y_train)
    else:
        search.fit(X_train, y_train, groups=fit_groups)
    return search.best_estimator_, search.best_params_, search.best_score_


def entrenar_xgb(X_train, y_train, groups=None):
    if not HAS_XGB:
        raise RuntimeError('xgboost no disponible')

    le = LabelEncoder()
    y_enc = le.fit_transform(pd.Series(y_train).astype(str))

    base = XGBClassifier(
        objective='multi:softprob',
        eval_metric='mlogloss',
        random_state=RANDOM_SEED,
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        tree_method='hist',
        num_class=len(le.classes_),
    )

    if not USE_RANDOMIZED_SEARCH:
        base.fit(X_train, y_enc)
        return base, {'modo': 'sin_tuning'}, np.nan, le

    cv, fit_groups = _resolve_cv(groups)

    param_dist = {
        'n_estimators': [200, 400, 700],
        'learning_rate': [0.03, 0.05, 0.1],
        'max_depth': [3, 5, 7, 9],
        'subsample': [0.7, 0.9, 1.0],
        'colsample_bytree': [0.7, 0.9, 1.0],
        'min_child_weight': [1, 3, 5],
        'reg_lambda': [0.5, 1.0, 2.0],
    }
    search = RandomizedSearchCV(
        base,
        param_distributions=param_dist,
        n_iter=N_ITER_SEARCH,
        scoring='f1_macro',
        cv=cv,
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS,
        verbose=1,
    )
    if fit_groups is None:
        search.fit(X_train, y_enc)
    else:
        search.fit(X_train, y_enc, groups=fit_groups)
    return search.best_estimator_, search.best_params_, search.best_score_, le


def evaluar_y_exportar(y_true, y_pred, titulo: str, prefijo: str):
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    bacc = balanced_accuracy_score(y_true, y_pred)
    labels = sorted(pd.Series(list(set(y_true) | set(y_pred))).unique())

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_norm = confusion_matrix(y_true, y_pred, labels=labels, normalize='true')

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm_norm)
    ax.set_xticks(range(len(labels)), labels=labels, rotation=45, ha='right')
    ax.set_yticks(range(len(labels)), labels=labels)
    ax.set_xlabel('Predicción')
    ax.set_ylabel('Real')
    ax.set_title(titulo)
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha='center', va='center')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'{prefijo}_confusion.png', dpi=200)
    plt.close(fig)

    report = classification_report(y_true, y_pred, digits=4, output_dict=True)
    pd.DataFrame(report).transpose().to_csv(OUT_DIR / f'reporte_{prefijo}.csv')

    return macro_f1, bacc


In [6]:
def _extraer_metricas_clase(report: dict, clase: str):
    sec = report.get(clase, {}) or {}
    return float(sec.get('f1-score', np.nan)), float(sec.get('support', np.nan))


def ejecutar_experimentos(profile: str, X_tr, y_tr, G_tr, X_eval, y_eval, M_eval):
    filas = []
    modelos = {}

    # RandomForest
    if 'rf' in MODELS_SELECTED:
        rf, rf_params, rf_cv = entrenar_rf(X_tr, y_tr, groups=G_tr)
        yhat_rf = rf.predict(X_eval)
        f1_rf, bacc_rf = evaluar_y_exportar(y_eval, yhat_rf, f'{profile} | RandomForest', f'{profile}_rf_{EVAL_ON}')

        pred_rf = M_eval.copy()
        pred_rf['y_pred'] = yhat_rf
        if hasattr(rf, 'predict_proba'):
            probs = rf.predict_proba(X_eval)
            for i, cls in enumerate(rf.classes_):
                pred_rf[f'prob_{cls}'] = probs[:, i]
        pred_rf.to_csv(OUT_DIR / f'predicciones_{profile}_rf_{EVAL_ON}.csv', index=False)

        rep_rf = classification_report(y_eval, yhat_rf, digits=4, output_dict=True)
        f1_a, sup_a = _extraer_metricas_clase(rep_rf, 'ansiedad')
        f1_d, sup_d = _extraer_metricas_clase(rep_rf, 'depresion')

        filas.append({
            'variant_name': VARIANT_NAME,
            'profile': profile,
            'model': 'RF',
            'cv_macro_f1': rf_cv,
            'macro_f1': f1_rf,
            'balanced_acc': bacc_rf,
            'precision_macro': float(rep_rf.get('macro avg', {}).get('precision', np.nan)),
            'recall_macro': float(rep_rf.get('macro avg', {}).get('recall', np.nan)),
            'f1_ansiedad': f1_a,
            'f1_depresion': f1_d,
            'soporte_ansiedad': sup_a,
            'soporte_depresion': sup_d,
            'best_params': json.dumps(rf_params, ensure_ascii=False),
            'n_features': X_tr.shape[1],
            'n_train': len(X_tr),
            'n_eval': len(X_eval),
            'eval_split': EVAL_ON,
            'seed': TRAIN_SEED,
            'run_id_features_core': FEATURE_RUN_ID_CORE,
            'run_id_features_py': FEATURE_RUN_ID_PY,
            'llm_activo': int(TRAIN_USE_LLM and FEATURE_LLM_ACTIVO),
            'sentimiento_activo': int(FEATURE_SENT_ACTIVO and TRAIN_USE_SENTIMENT),
            'beto_activo': int(TRAIN_USE_CONTEXT),
            'contexto_activo': int(TRAIN_USE_CONTEXT),
            'text_backbone': FEATURE_TEXT_BACKBONE,
            'context_prefixes': '|'.join(FEATURE_CONTEXT_PREFIXES),
            'template_activo': int(TRAIN_USE_TEMPLATE),
            'feat_activo': int(TRAIN_USE_FEAT),
            'reglas_activas': int(TRAIN_USE_RULES),
            'medicacion_activa': int(TRAIN_USE_MEDICATION),
        })
        modelos['RF'] = rf

    # XGBoost
    if 'xgb' in MODELS_SELECTED:
        if HAS_XGB:
            xgb, xgb_params, xgb_cv, le_xgb = entrenar_xgb(X_tr, y_tr, groups=G_tr)
            yhat_raw = np.asarray(xgb.predict(X_eval))

            # Compatibilidad entre versiones de XGBoost:
            # algunas devuelven índices de clase (1D) y otras probabilidades (2D).
            if yhat_raw.ndim == 2:
                yhat_idx = np.argmax(yhat_raw, axis=1)
            else:
                yhat_idx = yhat_raw.astype(int)

            yhat_xgb = le_xgb.inverse_transform(np.asarray(yhat_idx).astype(int))
            f1_xgb, bacc_xgb = evaluar_y_exportar(y_eval, yhat_xgb, f'{profile} | XGBoost', f'{profile}_xgb_{EVAL_ON}')

            pred_xgb = M_eval.copy()
            pred_xgb['y_pred'] = yhat_xgb

            probs = None
            if hasattr(xgb, 'predict_proba'):
                probs = np.asarray(xgb.predict_proba(X_eval))
            elif yhat_raw.ndim == 2:
                probs = yhat_raw

            if probs is not None and probs.ndim == 2:
                n_cols = min(probs.shape[1], len(le_xgb.classes_))
                for i in range(n_cols):
                    cls = le_xgb.classes_[i]
                    pred_xgb[f'prob_{cls}'] = probs[:, i]

            pred_xgb.to_csv(OUT_DIR / f'predicciones_{profile}_xgb_{EVAL_ON}.csv', index=False)

            rep_xgb = classification_report(y_eval, yhat_xgb, digits=4, output_dict=True)
            f1_a, sup_a = _extraer_metricas_clase(rep_xgb, 'ansiedad')
            f1_d, sup_d = _extraer_metricas_clase(rep_xgb, 'depresion')

            filas.append({
                'variant_name': VARIANT_NAME,
                'profile': profile,
                'model': 'XGB',
                'cv_macro_f1': xgb_cv,
                'macro_f1': f1_xgb,
                'balanced_acc': bacc_xgb,
                'precision_macro': float(rep_xgb.get('macro avg', {}).get('precision', np.nan)),
                'recall_macro': float(rep_xgb.get('macro avg', {}).get('recall', np.nan)),
                'f1_ansiedad': f1_a,
                'f1_depresion': f1_d,
                'soporte_ansiedad': sup_a,
                'soporte_depresion': sup_d,
                'best_params': json.dumps(xgb_params, ensure_ascii=False),
                'n_features': X_tr.shape[1],
                'n_train': len(X_tr),
                'n_eval': len(X_eval),
                'eval_split': EVAL_ON,
                'seed': TRAIN_SEED,
                'run_id_features_core': FEATURE_RUN_ID_CORE,
                'run_id_features_py': FEATURE_RUN_ID_PY,
                'llm_activo': int(TRAIN_USE_LLM and FEATURE_LLM_ACTIVO),
                'sentimiento_activo': int(FEATURE_SENT_ACTIVO and TRAIN_USE_SENTIMENT),
                'beto_activo': int(TRAIN_USE_CONTEXT),
                'contexto_activo': int(TRAIN_USE_CONTEXT),
                'text_backbone': FEATURE_TEXT_BACKBONE,
                'context_prefixes': '|'.join(FEATURE_CONTEXT_PREFIXES),
                'template_activo': int(TRAIN_USE_TEMPLATE),
                'feat_activo': int(TRAIN_USE_FEAT),
                'reglas_activas': int(TRAIN_USE_RULES),
                'medicacion_activa': int(TRAIN_USE_MEDICATION),
            })
            modelos['XGB'] = xgb
        else:
            print('Aviso: Se omite XGBoost porque no está instalado en el entorno.')

    return filas, modelos


filas_totales = []
modelos_por_profile = {}
cols_por_profile = {}

for profile in PROFILES_SELECTED:
    pdata = profile_data[profile]
    X_eval = pdata['X_dev'] if EVAL_ON == 'dev' else pdata['X_te']
    y_eval = pdata['y_dev'] if EVAL_ON == 'dev' else pdata['y_te']
    M_eval = pdata['M_dev'] if EVAL_ON == 'dev' else pdata['M_te']

    filas_p, modelos_p = ejecutar_experimentos(
        profile=profile,
        X_tr=pdata['X_tr'],
        y_tr=pdata['y_tr'],
        G_tr=pdata['G_tr'],
        X_eval=X_eval,
        y_eval=y_eval,
        M_eval=M_eval,
    )
    filas_totales.extend(filas_p)
    modelos_por_profile[profile] = modelos_p
    cols_por_profile[profile] = list(pdata['X_tr'].columns)

df_metricas = pd.DataFrame(filas_totales)
metricas_path = OUT_DIR / f'comparacion_modelos_{EVAL_ON}.csv'
df_metricas.to_csv(metricas_path, index=False)
print('Guardado:', metricas_path)

# Ablación por perfil léxico (`core` vs `py`) cuando ambos están presentes
if not df_metricas.empty:
    abl = df_metricas.pivot_table(index='model', columns='profile', values='macro_f1', aggfunc='mean').reset_index()
    if {'core', 'py'}.issubset(set(abl.columns)):
        abl['delta_py_vs_core'] = abl['py'] - abl['core']
    abl_path = OUT_DIR / f'ablacion_perfiles_{EVAL_ON}.csv'
    abl.to_csv(abl_path, index=False)
    print('Guardado:', abl_path)

display(df_metricas.sort_values(['macro_f1', 'balanced_acc'], ascending=False))


Fitting 3 folds for each of 25 candidates, totalling 75 fits
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Guardado: /Users/manuelnunez/Projects/psych-phenotyping-paraguay/data/outputs/train_20260310_093418/comparacion_modelos_dev.csv
Guardado: /Users/manuelnunez/Projects/psych-phenotyping-paraguay/data/outputs/train_20260310_093418/ablacion_perfiles_dev.csv


,profile,model,cv_macro_f1,macro_f1,balanced_acc,best_params,n_features,n_train,n_eval,eval_split
3,py,XGB,NaN,0.688997,0.684753,"{""subsample"": 1.0, ""reg_lambda"": 1.0, ""n_estim...",958,1107,343,dev
1,core,XGB,NaN,0.681765,0.677695,"{""subsample"": 1.0, ""reg_lambda"": 1.0, ""n_estim...",954,1107,343,dev
0,core,RF,0.656522,0.677252,0.687716,"{""n_estimators"": 300, ""min_samples_split"": 2, ...",954,1107,343,dev
2,py,RF,0.659403,0.673086,0.682716,"{""n_estimators"": 300, ""min_samples_split"": 2, ...",958,1107,343,dev


In [7]:
# Guardado de modelos y columnas de características

def guardar_bundle(modelos: dict, x_cols: list[str], prefijo: str):
    for nombre, modelo in modelos.items():
        model_path = OUT_DIR / f'modelo_{prefijo}_{nombre}.joblib'
        joblib.dump(modelo, model_path)
        print('Modelo:', model_path.name)

    cols_path = OUT_DIR / f'{prefijo}_X_cols.json'
    with open(cols_path, 'w', encoding='utf-8') as f:
        json.dump(x_cols, f, ensure_ascii=False, indent=2)
    print('Columnas de features:', cols_path.name)


for profile, modelos in modelos_por_profile.items():
    guardar_bundle(modelos, cols_por_profile.get(profile, []), profile)

resumen = {
    'run_id': RUN_ID,
    'variant_name': VARIANT_NAME,
    'eval_on': EVAL_ON,
    'feature_run_id_core': FEATURE_RUN_ID_CORE,
    'feature_run_id_py': FEATURE_RUN_ID_PY,
    'feature_llm_activo': FEATURE_LLM_ACTIVO,
    'feature_sentimiento_activo': FEATURE_SENT_ACTIVO,
    'feature_contexto_activo': FEATURE_CONTEXT_ACTIVO,
    'feature_text_backbone': FEATURE_TEXT_BACKBONE,
    'feature_context_prefixes': FEATURE_CONTEXT_PREFIXES,
    'profiles_selected': PROFILES_SELECTED,
    'models_selected': MODELS_SELECTED,
    'use_randomized_search': USE_RANDOMIZED_SEARCH,
    'n_iter_search': N_ITER_SEARCH,
    'cv_folds': CV_FOLDS,
    'seed': TRAIN_SEED,
    'has_xgb': HAS_XGB,
    'use_xgb': USE_XGB,
    'require_xgb': REQUIRE_XGB,
    'train_drop_columns': TRAIN_DROP_COLUMNS,
    'train_drop_prefixes': TRAIN_DROP_PREFIXES,
    'train_keep_prefixes': TRAIN_KEEP_PREFIXES,
    'train_use_llm': TRAIN_USE_LLM,
    'train_use_beto': TRAIN_USE_CONTEXT,
    'train_use_context': TRAIN_USE_CONTEXT,
    'train_use_template': TRAIN_USE_TEMPLATE,
    'train_use_feat': TRAIN_USE_FEAT,
    'train_use_rules': TRAIN_USE_RULES,
    'train_use_medication': TRAIN_USE_MEDICATION,
    'train_use_sentiment': TRAIN_USE_SENTIMENT,
}
with open(OUT_DIR / 'resumen_entrenamiento.json', 'w', encoding='utf-8') as f:
    json.dump(resumen, f, ensure_ascii=False, indent=2)

# Resumen específico de ablación
res_abl = {
    'run_id': RUN_ID,
    'variant_name': VARIANT_NAME,
    'profiles': PROFILES_SELECTED,
    'models': MODELS_SELECTED,
    'flags': {
        'llm_activo': bool(TRAIN_USE_LLM and FEATURE_LLM_ACTIVO),
        'sentimiento_activo_feature': FEATURE_SENT_ACTIVO,
        'use_beto': TRAIN_USE_CONTEXT,
        'use_context': TRAIN_USE_CONTEXT,
        'use_template': TRAIN_USE_TEMPLATE,
        'use_feat': TRAIN_USE_FEAT,
        'use_rules': TRAIN_USE_RULES,
        'use_medication': TRAIN_USE_MEDICATION,
        'use_sentiment': TRAIN_USE_SENTIMENT,
    },
    'drop_columns': TRAIN_DROP_COLUMNS,
    'drop_prefixes': TRAIN_DROP_PREFIXES,
    'keep_prefixes': TRAIN_KEEP_PREFIXES,
    'seed': TRAIN_SEED,
    'feature_logs_path': str(OUT_DIR / 'detalle_columnas_ablacion.json'),
}
with open(OUT_DIR / 'resumen_ablacion.json', 'w', encoding='utf-8') as f:
    json.dump(res_abl, f, ensure_ascii=False, indent=2)

print('Entrenamiento híbrido finalizado.')
print('Carpeta de salida:', OUT_DIR)


Modelo: modelo_core_RF.joblib
Modelo: modelo_core_XGB.joblib
Columnas de features: core_X_cols.json
Modelo: modelo_py_RF.joblib
Modelo: modelo_py_XGB.joblib
Columnas de features: py_X_cols.json
Entrenamiento híbrido finalizado.
Carpeta de salida: /Users/manuelnunez/Projects/psych-phenotyping-paraguay/data/outputs/train_20260310_093418


## Balance, desbalance y criterio de evaluación

En el corte actual la tarea sigue siendo binaria (`ansiedad`, `depresion`) y el dataset final queda **claramente desbalanceado** a favor de `depresion`.

Para no leer el problema solo desde accuracy:

- se priorizan `macro_f1`, `balanced_accuracy` y F1 por clase;
- `RandomForest` se entrena con `class_weight='balanced'`;
- `XGBoost` se mantiene sin reponderación explícita adicional en la versión vigente;
- no se aplicó sampling/oversampling en este pipeline de desarrollo.

Esta combinación permite comparar variantes sin ocultar que `ansiedad` es la clase más difícil en el corte actual.
